In [ ]:
### 결정 트리(Descision Tree)



# pandas 라이브러리로 외부 데이터 호출
import pandas as pd

wine = pd.read_csv('https://bit.ly/wine_csv_data')
wine.head()

wine.info() #각 데이터 열의 타입과 누락 여부를 제공하는 pandas 메서드

wine.describe() # 각 데이터 열에 대한 간략한 통계를 제공하는 pandas 메서드

# wine 데이터프레임으로부터 입력과 타깃 데이터 추출
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']



# data/target으로부터 훈련-테스트 데이터 추출(입력-타깃)
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)
# test_size 파라미터를 설정함으로써 훈련-테스트 개수 비율을 조정할 수 있다. 0.2 = 20%

# 실제로 타깃 개수와 비율을 관찰하면 테스트 타깃 비율이 20%에 수렴한다.
print(f"훈련용 타깃 개수: {train_target.shape}, 테스트 타깃 개수{test_target.shape}")
print(f"전체 타깃 개수에 대한 테스트 타깃 개수 비: {test_target.shape[0] / target.shape[0]}")



# 데이터 전처리 - 정규화 진행
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)



## 로지스틱 회귀로 와인 분류(레드 or 화이트 - 이진 분류)

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(C = 1)
lr.fit(train_scaled, train_target)
# 예상보다 낮은 로지스틱 회귀 정확도 - 과소적합 (note. 규제 정도 C를 수정해도 결과 유지되는 상황.)
print(lr.score(train_scaled, train_target))
print(lr.score(test_scaled, test_target))
print(lr.coef_, lr.intercept_)



# 한 가지 의문점: ML 알고리즘(로지스틱 회귀 등)을 타인에게 설득할 수 있는가?
# => 내부 작동(모델 파라미터 연산)을 알 수 없으므로 어렵다. => 더 직관적인 모델이 필요하다.
# 더 직관적인 결정 트리(Descision Tree) 알고리즘 사용

from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(train_scaled, train_target)
print(dt.score(train_scaled, train_target))
print(dt.score(test_scaled, test_target)) # 결정 트리 알고리즘 1차 실행 결과 - 과대적합

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree # 결정 트리 알고리즘 결과를 시각화하는 메서드 plot_tree
#plt.figure(figsize=(10,7)) # figure 메서드 => 새로운 맷플롯립 그림 객체 생성 + figsize로 크기 결정(인치)
#plot_tree(dt) # 컴퓨터 환경에 따라 처리 시간 up
#plt.show()



# 모든 결정 트리 노드를 시각화 => 복잡하다(직관성 down) => 가지치기 필요(깊이 제한)

dt = DecisionTreeClassifier(max_depth=3, random_state=42) # 노드 깊이 = 3으로 제한
dt.fit(train_scaled, train_target)
# 가지치기 결과 => 훈련 정확도 감소 & 테스트 정확도 보합 => 과소적합?
print(dt.score(train_scaled, train_target))
print(dt.score(test_scaled, test_target))

plt.figure(figsize=(20, 15))
plot_tree(dt, filled=True, feature_names=['alcohol', 'sugar', 'pH']) # 노드별 색 차이 + 특징명 반영한 plot_tree
plt.show()



# 결정 트리 => 정보 이득(불순도)에 따라 예측 결정 => 특성 간 영향 X => 데이터 표준화 불필요(확인 절차)
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(train_input, train_target)
# 표준화한 79-80행(train_scaled와 test_scaled를 사용한 알고리즘)과 정확도 동일.
print(dt.score(train_input, train_target))
print(dt.score(test_input, test_target))

# 데이터 원본으로 결정 트리 학습 => 사용자가 판별 기준을 직관적으로 이해하기 쉽다(노드에 표현된 값 = 실제 값)
plt.figure(figsize=(20, 15))
plot_tree(dt, filled=True, feature_names=['alcohol', 'sugar', 'pH'])
plt.show()

print(dt.feature_importances_) # 특성별 중요도 출력

In [ ]:
# 샘플 수에 따른 결정 트리 알고리즘 VS 선형 회귀 알고리즘 계산 복잡도 비교

import matplotlib.pyplot as plt
import numpy as np

k = 10 # 특성 k = 10개로 고정
n_list = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100]) # 특성 개수 n을 리스트 형태로 저장

Decision_Tree_result = []
Linear_Regression_result = []

for n in n_list:
  Decision_Tree_result.append(k * n *np.log2(n)) # 로그함수는 외부 라이브러리(넘파이) 메서드 사용
  Linear_Regression_result.append(k**3) #(역행렬 연산 기준)

plt.plot(n_list, Decision_Tree_result, label='Decision Tree')
plt.plot(n_list, Linear_Regression_result, label = 'Linear Regression')
plt.xlabel("The amount of n(samples)")
plt.ylabel("Computing Complexity")
plt.legend()
plt.grid(True)
plt.show()
